# 🔬 RevertIQ — Backtesting & Performance

This notebook demonstrates:
1. Running the full portfolio backtest
2. Analysing performance metrics
3. Equity curve, drawdown, and trade analysis
4. Walk-forward parameter optimisation
5. Sensitivity analysis

In [ ]:
import warnings; warnings.filterwarnings('ignore')

from revertiq.config.settings import get_default_config
from revertiq.data.downloader import DataDownloader
from revertiq.data.cleaner import DataCleaner
from revertiq.indicators.technical import TechnicalIndicators
from revertiq.features.engineering import FeatureEngineer
from revertiq.signals.regime import RegimeFilter
from revertiq.signals.generator import SignalGenerator
from revertiq.ranking.ranker import StockRanker
from revertiq.backtesting.engine import BacktestEngine
from revertiq.backtesting.metrics import PerformanceMetrics
from revertiq.backtesting.optimizer import WalkForwardOptimizer
from revertiq.visualization.charts import ChartEngine
from revertiq.visualization.reports import ReportGenerator

import pandas as pd
import numpy as np
from dataclasses import replace

config = get_default_config()
ce = ChartEngine()
rg = ReportGenerator()
print("Ready ✓")

## 1. Prepare All Data

In [ ]:
# Full data pipeline (download if needed)
dl = DataDownloader(config.data)
cleaner = DataCleaner(config.data)

try:
    clean_data = cleaner.load_all_processed()
except FileNotFoundError:
    dl.download_all()
    clean_data = cleaner.clean_all()

combined = cleaner.get_combined_df()
index_data = dl.download_index_data()

nifty_data = index_data.get('nifty50', pd.DataFrame())
if not nifty_data.empty:
    nifty_data.columns = [c.lower() for c in nifty_data.columns]
vix_data = index_data.get('india_vix', None)
if vix_data is not None and not vix_data.empty:
    vix_data.columns = [c.lower() for c in vix_data.columns]

# Indicators
ti = TechnicalIndicators()
enriched = [ti.compute_all(combined[combined['ticker']==t].copy().sort_values('date'), config.indicator)
            for t in combined['ticker'].unique()]
stock_data = pd.concat(enriched, ignore_index=True)

# Features
fe = FeatureEngineer(config.indicator, config.ranking)
features_df = fe.compute_all_features(stock_data, nifty_data)

# Regime
rf = RegimeFilter(config.regime)
nifty_idx = nifty_data.set_index('date') if 'date' in nifty_data.columns else nifty_data
regime_series = rf.compute_regime(nifty_idx, vix_data)

# Signals
sg = SignalGenerator(config.signal)
signals_df = sg.generate_all_signals(features_df, regime_series)

# Rankings
ranker = StockRanker(config.ranking)
rankings_df = ranker.rank_all_dates(features_df, signals_df)

print(f"Pipeline complete. Signals: {signals_df['buy_signal'].sum()} buys")

## 2. Run Backtest

In [ ]:
# Prepare data
bt_data = features_df.copy()
if 'date' not in bt_data.columns:
    bt_data = bt_data.reset_index()

benchmark = nifty_data.copy()
if 'date' not in benchmark.columns:
    benchmark = benchmark.reset_index()

# Run backtest
engine = BacktestEngine(config)
result = engine.run_with_benchmark(
    price_data=bt_data,
    signals_df=signals_df,
    rankings_df=rankings_df,
    regime_series=regime_series,
    benchmark_data=benchmark,
)

# Print summary
rg.print_metrics_summary(result.metrics)

## 3. Equity Curve

In [ ]:
fig = ce.equity_curve(result.equity_curve, result.benchmark_equity)
fig.show()

## 4. Drawdown Analysis

In [ ]:
dd = PerformanceMetrics.drawdown_series(result.equity_curve)
if dd is not None and not dd.empty:
    fig = ce.drawdown_chart(dd)
    fig.show()
    
    # Drawdown stats
    dd_info = PerformanceMetrics.max_drawdown(result.equity_curve)
    if dd_info:
        print(f"Max Drawdown  : {dd_info[0]*100:.2f}%")
        print(f"Peak date     : {dd_info[1]}")
        print(f"Trough date   : {dd_info[2]}")

## 5. Rolling Sharpe

In [ ]:
if not result.daily_returns.empty:
    rs = PerformanceMetrics.rolling_sharpe(result.daily_returns)
    if rs is not None and not rs.empty:
        fig = ce.rolling_sharpe_chart(rs)
        fig.show()

## 6. Monthly Returns

In [ ]:
if not result.daily_returns.empty:
    monthly = PerformanceMetrics.monthly_returns(result.daily_returns)
    if monthly is not None and not monthly.empty:
        fig = ce.monthly_returns_heatmap(monthly)
        fig.show()

## 7. Return Distribution

In [ ]:
if not result.daily_returns.empty:
    fig = ce.return_distribution(result.daily_returns)
    fig.show()

## 8. Trade Analysis

In [ ]:
tl = result.trade_log
if not tl.empty:
    print(f"Total trades: {len(tl)}")
    print(f"Winners: {(tl['pnl']>0).sum()} | Losers: {(tl['pnl']<=0).sum()}")
    print(f"Win rate: {(tl['pnl']>0).mean()*100:.1f}%")
    print(f"Avg PnL: Rs.{tl['pnl'].mean():,.0f}")
    print(f"Avg holding: {tl['holding_days'].mean():.1f} days")
    print(f"\nExit reasons:")
    print(tl['exit_reason'].value_counts().to_string())
    
    # P&L distribution
    import plotly.express as px
    fig = px.histogram(tl, x='pnl', nbins=40, title='Trade P&L Distribution',
                       color_discrete_sequence=['#58a6ff'])
    fig.add_vline(x=0, line_dash='dash', line_color='#f85149')
    fig.update_layout(template='plotly_dark',
                      paper_bgcolor='#161b22', plot_bgcolor='#0d1117')
    fig.show()
    
    # Show last 20 trades
    print("\nLast 20 trades:")
    print(tl.tail(20).to_string(index=False))
else:
    print("No trades executed. Try adjusting signal thresholds.")

## 9. Sensitivity Analysis

In [ ]:
# Test with different transaction cost assumptions
cost_scenarios = [0.0, 0.001, 0.002, 0.003]
cost_results = []

for cost in cost_scenarios:
    test_config = get_default_config()
    test_config.backtest = replace(test_config.backtest,
                                   buy_cost_pct=cost/2, sell_cost_pct=cost/2)
    eng = BacktestEngine(test_config)
    res = eng.run(bt_data, signals_df, rankings_df, regime_series)
    m = res.metrics
    cost_results.append({
        'cost_pct': f"{cost*100:.1f}%",
        'cagr': m.get('cagr', 0),
        'sharpe': m.get('sharpe_ratio', 0),
        'trades': m.get('total_trades', 0),
    })

print("\nTransaction Cost Sensitivity:")
print(pd.DataFrame(cost_results).to_string(index=False))

## 10. Walk-Forward Optimisation

In [ ]:
# Walk-forward parameter optimisation
# NOTE: This can take several minutes depending on grid size

# Use smaller grid for demo
from dataclasses import replace as dr
opt_config = get_default_config()
opt_config.optimization = dr(opt_config.optimization, param_grid={
    'rsi_buy_threshold': [5, 10, 15],
    'zscore_buy_threshold': [-2.0, -1.5],
    'max_holding_days': [5, 10],
})

optimizer = WalkForwardOptimizer(opt_config)
wfo_result = optimizer.optimize(bt_data, nifty_data)
# The optimizer prints a formatted summary automatically

## 11. Export Report

In [ ]:
# Generate full HTML report
report_path = rg.generate_backtest_report(result, output_dir='data/results')
print(f"Report saved to: {report_path}")

# Export CSV files
if not tl.empty:
    rg.export_trade_log(tl)
if not result.equity_curve.empty:
    rg.export_equity_curve(result.equity_curve)

print("\nAll exports complete ✓")